# Copy training data, excluding bad tiles

Walks each source directory (same recursive walk as `HMATensorDataset` --
including the nested lidar-point-count subfolders), scans every `.npy`
tensor for the same problems the QC notebook found, and **copies only the
good files** to a new location, preserving the exact folder structure.

Bad-file criteria (all configurable below):
- Fails to load, or wrong shape (not 4 channels).
- Contains NaN/Inf.
- Mask channel isn't actually binary.
- Elevation outside a plausible range.
- **`dem_bic` void/corruption** -- the main issue found in QC: `residual_mean`
  (`gt_dem - dem_bic` averaged over the whole tile) is a big outlier, or
  `dem_min` is suspiciously close to 0 (void sentinel). This is what was
  driving the huge training loss.

Nothing is deleted or modified at the source -- this only copies the files
that pass into `DEST_ROOT`, mirroring the source folder structure. Point
your training notebook's `TRAIN_DIRS`/`VAL_DIRS` at `DEST_ROOT/<name>` once
you're happy with the result.

Set `DRY_RUN = True` (default) to see counts and a manifest without copying
anything yet -- flip to `False` once the numbers look right.


In [23]:
import os, shutil, json, csv
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm


In [24]:
def load_hma_npy(path):
    """Load one HMA tensor .npy file, tolerant of two container formats:

    Legacy: a plain (4, H, W) float array -- dem_bic, lidar_raw, mask, gt_dem.

    Extended: the same 4 image channels plus per-file georeferencing
    metadata appended after them --
        Ch5: GDAL geotransform, shape (6,) float64, (x0, dx, 0, y0, 0, -dy)
        Ch6: EPSG code, single int.
    These extra fields do not share the (H, W) shape of the image channels,
    so they can only be packed as a heterogeneous container (object-dtype
    array wrapping a list/tuple, or a dict) -- this auto-detects which.

    Returns (image, geotransform, epsg):
        image        -- (4, H, W) float32 ndarray
        geotransform -- (6,) float64 ndarray, or None
        epsg         -- int, or None
    """
    try:
        raw = np.load(path, allow_pickle=False)
    except ValueError:
        raw = np.load(path, allow_pickle=True)

    if raw.dtype != object and raw.ndim == 3 and raw.shape[0] == 4:
        return raw.astype(np.float32, copy=False), None, None

    obj = raw
    if isinstance(raw, np.ndarray) and raw.dtype == object:
        obj = raw.item() if raw.shape == () else raw

    if isinstance(obj, dict):
        def _get(*names):
            for n in names:
                if n in obj:
                    return obj[n]
            return None
        dem_bic      = _get("dem_bic", "Ch1", "ch1")
        lidar_raw    = _get("lidar_raw", "Ch2", "ch2")
        mask         = _get("mask", "Ch3", "ch3")
        gt_dem       = _get("gt_dem", "Ch4", "ch4")
        geotransform = _get("geotransform", "gt", "affine", "Ch5", "ch5")
        epsg         = _get("epsg", "EPSG", "Ch6", "ch6")
    elif isinstance(obj, (list, tuple, np.ndarray)) and len(obj) >= 4:
        dem_bic, lidar_raw, mask, gt_dem = obj[0], obj[1], obj[2], obj[3]
        geotransform = obj[4] if len(obj) > 4 else None
        epsg = obj[5] if len(obj) > 5 else None
    else:
        raise ValueError(
            "Unrecognized .npy container for " + str(path) + ": dtype=" + str(raw.dtype) +
            ", shape=" + str(getattr(raw, "shape", None)) + ", type=" + str(type(obj))
        )

    image = np.stack([
        np.asarray(dem_bic, dtype=np.float32),
        np.asarray(lidar_raw, dtype=np.float32),
        np.asarray(mask, dtype=np.float32),
        np.asarray(gt_dem, dtype=np.float32),
    ], axis=0)
    geotransform = np.asarray(geotransform, dtype=np.float64) if geotransform is not None else None
    epsg = int(epsg) if epsg is not None else None
    return image, geotransform, epsg


In [25]:
# ============================================================================
# CONFIG -- edit for your dataset
# ============================================================================

# Source directories to walk (each becomes a top-level folder under DEST_ROOT
# with the same name, structure preserved beneath it).
SOURCE_DIRS = [
    r'D:\Vaibhav\dem-lidar\dataset_12m\tensors\train',
    r'D:\Vaibhav\dem-lidar\tensors_256',
]

# Where the cleaned copy goes. Will contain DEST_ROOT/train/... and
# DEST_ROOT/validation_contiguous/... mirroring the sources above.
DEST_ROOT = Path(r'D:\Vaibhav\dem-lidar\dataset_12m_clean\tensors')

EXPECTED_CHANNELS = 4
ELEV_MIN_PLAUSIBLE = -500.0
ELEV_MAX_PLAUSIBLE = 9000.0

# The two checks that caught the real problem last time:
RESIDUAL_ABS_THRESHOLD = 50.0   # |mean(gt_dem - dem_bic)| over the whole tile, metres
DEM_MIN_VOID_THRESHOLD = 50.0   # dem_bic.min() below this -> likely void/no-data sentinel
MASK_TOLERANCE = 1e-6

DRY_RUN = True   # True: scan + report only, no files copied. Flip to False when ready.

for d in SOURCE_DIRS:
    if not Path(d).exists():
        print(f'WARNING: source path does not exist -- {d}')

print(f'Sources: {SOURCE_DIRS}')
print(f'Destination root: {DEST_ROOT}')
print(f'DRY_RUN = {DRY_RUN}')


Sources: ['D:\\Vaibhav\\dem-lidar\\dataset_12m\\tensors\\train', 'D:\\Vaibhav\\dem-lidar\\tensors_256']
Destination root: D:\Vaibhav\dem-lidar\dataset_12m_clean\tensors
DRY_RUN = True


In [26]:
# ============================================================================
# SCAN -- decide keep/exclude for every file, without moving anything yet
# ============================================================================
results = []   # dicts: {source_dir, path, rel_path, keep, reasons}

for source_dir in SOURCE_DIRS:
    source_root = Path(source_dir)
    top_name = source_root.name

    filepaths = []
    for root, _, files in os.walk(source_root):
        for f in files:
            if f.endswith('.npy') or f.endswith('npz'):
                filepaths.append(Path(root) / f)

    for path in tqdm(filepaths, desc=f'Scanning {top_name}'):
        reasons = []
        rel_path = path.relative_to(source_root)

        try:
            data, geotransform, epsg = load_hma_npy(path)
        except Exception as e:
            results.append({
                'source_dir': str(source_dir), 'top_name': top_name,
                'path': str(path), 'rel_path': str(rel_path),
                'keep': False, 'reasons': [f'load_error: {e}'],
            })
            continue

        if data.ndim != 3 or data.shape[0] != EXPECTED_CHANNELS:
            results.append({
                'source_dir': str(source_dir), 'top_name': top_name,
                'path': str(path), 'rel_path': str(rel_path),
                'keep': False, 'reasons': [f'wrong_shape: {data.shape}'],
            })
            continue

        dem_bic, lidar_raw, mask, gt_dem = data[0], data[1], data[2], data[3]

        if not np.all(np.isfinite(data)):
            reasons.append('nan_inf')

        mask_vals = mask[np.isfinite(mask)]
        if mask_vals.size > 0:
            is_binary = np.all(
                (np.abs(mask_vals) < MASK_TOLERANCE) | (np.abs(mask_vals - 1.0) < MASK_TOLERANCE)
            )
            if not is_binary:
                reasons.append('non_binary_mask')

        dem_finite = np.isfinite(dem_bic)
        gt_finite = np.isfinite(gt_dem)
        dem_valid = dem_bic[dem_finite]
        gt_valid = gt_dem[gt_finite]

        if dem_valid.size > 0 and (dem_valid.min() < ELEV_MIN_PLAUSIBLE or dem_valid.max() > ELEV_MAX_PLAUSIBLE):
            reasons.append('implausible_elevation_dem_bic')
        if gt_valid.size > 0 and (gt_valid.min() < ELEV_MIN_PLAUSIBLE or gt_valid.max() > ELEV_MAX_PLAUSIBLE):
            reasons.append('implausible_elevation_gt_dem')

        # ---- the check that actually mattered last time ----
        both_finite = dem_finite & gt_finite
        if both_finite.sum() > 0:
            residual_mean = float((gt_dem - dem_bic)[both_finite].mean())
            if abs(residual_mean) > RESIDUAL_ABS_THRESHOLD:
                reasons.append(f'residual_outlier: {residual_mean:.1f}m')

        if dem_valid.size > 0 and dem_valid.min() < DEM_MIN_VOID_THRESHOLD:
            reasons.append(f'dem_bic_void_suspected: min={float(dem_valid.min()):.1f}m')

        results.append({
            'source_dir': str(source_dir), 'top_name': top_name,
            'path': str(path), 'rel_path': str(rel_path),
            'keep': len(reasons) == 0, 'reasons': reasons,
        })

n_total = len(results)
n_keep = sum(1 for r in results if r['keep'])
n_bad = n_total - n_keep
print(f'\nScanned {n_total} files -- keep {n_keep}, exclude {n_bad} ({n_bad/max(n_total,1)*100:.1f}%).')


Scanning train:   0%|          | 0/1428 [00:00<?, ?it/s]

Scanning tensors_256:   0%|          | 0/94 [00:00<?, ?it/s]


Scanned 1522 files -- keep 1428, exclude 94 (6.2%).


In [27]:
# ============================================================================
# EXCLUSION REPORT -- see why files were dropped before copying anything
# ============================================================================
from collections import Counter

reason_counts = Counter()
for r in results:
    for reason in r['reasons']:
        reason_counts[reason.split(':')[0]] += 1

print('Exclusion reasons (a file can have more than one):')
for reason, count in reason_counts.most_common():
    print(f'  {reason:35s} {count}')

print('\nPer-source breakdown:')
for source_dir in SOURCE_DIRS:
    top_name = Path(source_dir).name
    subset = [r for r in results if r['top_name'] == top_name]
    kept = sum(1 for r in subset if r['keep'])
    print(f'  {top_name:25s} {kept}/{len(subset)} kept')

print('\nWorst 10 excluded files:')
for r in [x for x in results if not x['keep']][:10]:
    print(f"  {r['path']}  ->  {r['reasons']}")


Exclusion reasons (a file can have more than one):
  load_error                          94

Per-source breakdown:
  train                     1428/1428 kept
  tensors_256               0/94 kept

Worst 10 excluded files:
  D:\Vaibhav\dem-lidar\tensors_256\00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00070.npz  ->  ["load_error: 'NpzFile' object has no attribute 'dtype'"]
  D:\Vaibhav\dem-lidar\tensors_256\00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00071.npz  ->  ["load_error: 'NpzFile' object has no attribute 'dtype'"]
  D:\Vaibhav\dem-lidar\tensors_256\00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00072.npz  ->  ["load_error: 'NpzFile' object has no attribute 'dtype'"]
  D:\Vaibhav\dem-lidar\tensors_256\00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00073.npz  ->  ["load_error: 'NpzFile' object has no attribute 'dtype'"]
  D:\Vaibhav\dem-l

In [28]:
# ============================================================================
# COPY -- only runs if DRY_RUN is False
# ============================================================================
if DRY_RUN:
    print('DRY_RUN is True -- nothing copied. Review the report above, then set DRY_RUN = False and re-run this cell.')
else:
    n_copied = 0
    n_failed = 0
    for r in tqdm([x for x in results if x['keep']], desc='Copying good files'):
        src = Path(r['path'])
        dst = DEST_ROOT / r['top_name'] / r['rel_path']
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copy2(src, dst)
            n_copied += 1
        except Exception as e:
            n_failed += 1
            tqdm.write(f'FAILED to copy {src}: {e}')

    print(f'\nCopied {n_copied} files to {DEST_ROOT}')
    if n_failed:
        print(f'{n_failed} files failed to copy -- see messages above.')


DRY_RUN is True -- nothing copied. Review the report above, then set DRY_RUN = False and re-run this cell.


In [29]:
# ============================================================================
# MANIFEST -- record exactly what was kept/excluded and why, for reproducibility
# ============================================================================
DEST_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = DEST_ROOT / 'copy_manifest.json'

with open(manifest_path, 'w', encoding='utf-8') as fh:
    json.dump({
        'source_dirs': SOURCE_DIRS,
        'dest_root': str(DEST_ROOT),
        'dry_run': DRY_RUN,
        'thresholds': {
            'residual_abs_threshold': RESIDUAL_ABS_THRESHOLD,
            'dem_min_void_threshold': DEM_MIN_VOID_THRESHOLD,
            'elev_min_plausible': ELEV_MIN_PLAUSIBLE,
            'elev_max_plausible': ELEV_MAX_PLAUSIBLE,
        },
        'n_total': n_total,
        'n_keep': n_keep,
        'n_excluded': n_bad,
        'files': results,
    }, fh, indent=1)

print(f'Manifest written to {manifest_path}')
print('Once copying is done, point your training notebook at:')
for source_dir in SOURCE_DIRS:
    top_name = Path(source_dir).name
    print(f'  {DEST_ROOT / top_name}')


Manifest written to D:\Vaibhav\dem-lidar\dataset_12m_clean\tensors\copy_manifest.json
Once copying is done, point your training notebook at:
  D:\Vaibhav\dem-lidar\dataset_12m_clean\tensors\train
  D:\Vaibhav\dem-lidar\dataset_12m_clean\tensors\tensors_256
